<a href="https://colab.research.google.com/github/Teixeiras15-collab/Atividades/blob/main/Semana7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Nível 1: Básico (URLs, Parâmetros e Cabeçalhos)

### Exercício 1.1, 1.2, 1.3 e 1.4: Requisição GET com Parâmetros e Cabeçalhos

In [1]:
import requests

# URL da API do JSONPlaceholder
url = 'https://jsonplaceholder.typicode.com/posts'

# Exercício 1.2: Parâmetros para filtrar os resultados (userId=2 e limit=5)
params = {
    'userId': 2,
    '_limit': 5  # Limita a quantidade de resultados
}

# Exercício 1.3: Cabeçalhos personalizados com User-Agent
headers = {
    'User-Agent': 'Meu Projeto Python/1.0 (Colab)',
    'Accept': 'application/json'
}

# Exercício 1.1 e 1.4: Fazendo a requisição GET com timeout
try:
    response = requests.get(url, params=params, headers=headers, timeout=10)

    # Exercício 1.4: Imprimindo o código de status HTTP e a URL final
    print(f"Status Code: {response.status_code}")
    print(f"URL Final: {response.url}")

    # Verificar se a requisição foi bem-sucedida (código 200)
    if response.status_code == 200:
        print("Requisição GET realizada com sucesso!")
        # Opcional: Imprimir uma parte do conteúdo para verificar os dados
        # print("Conteúdo (primeiros 500 caracteres):\n", response.text[:500])
    else:
        print(f"Erro na requisição. Código de status: {response.status_code}")

except requests.exceptions.Timeout:
    print("A requisição excedeu o tempo limite (timeout).")
except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro durante a requisição: {e}")


Status Code: 200
URL Final: https://jsonplaceholder.typicode.com/posts?userId=2&_limit=5
Requisição GET realizada com sucesso!


## Nível 2: Intermediário (JSON, Erros e Arquivos Binários)

### Exercício 2.1: Consulta de CEPs e Armazenamento em DataFrame

In [2]:
import requests
import pandas as pd

# Lista de três CEPs diferentes
ceps = ['01001000', '20040003', '70040010'] # Exemplos: Sé-SP, Centro-RJ, Asa Norte-DF

# Lista para armazenar os resultados das consultas
cep_data = []

# Loop para consultar cada CEP na API do ViaCEP
print("Consultando CEPs...")
for cep in ceps:
    url = f'https://viacep.com.br/ws/{cep}/json/'
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Lança exceções para códigos de status HTTP 4xx/5xx
        data = response.json()
        cep_data.append(data)
        print(f"CEP {cep} consultado com sucesso.")
    except requests.exceptions.Timeout:
        print(f"A requisição para o CEP {cep} excedeu o tempo limite.")
    except requests.exceptions.RequestException as e:
        print(f"Ocorreu um erro ao consultar o CEP {cep}: {e}")

# Criando um DataFrame do Pandas com os resultados
df_ceps = pd.DataFrame(cep_data)

print("\nDataFrame com os resultados:")
display(df_ceps)


Consultando CEPs...
CEP 01001000 consultado com sucesso.
CEP 20040003 consultado com sucesso.
CEP 70040010 consultado com sucesso.

DataFrame com os resultados:


,cep,logradouro,complemento,unidade,bairro,localidade,uf,estado,regiao,ibge,gia,ddd,siafi
0,01001-000,Praça da Sé,lado ímpar,,Sé,São Paulo,SP,São Paulo,Sudeste,3550308,1004,11,7107
1,20040-003,Avenida Rio Branco,de 146 ao fim - lado par,,Centro,Rio de Janeiro,RJ,Rio de Janeiro,Sudeste,3304557,,21,6001
2,70040-010,Quadra SBN Quadra 1,,,Asa Norte,Brasília,DF,Distrito Federal,Centro-Oeste,5300108,,61,9701


### Exercício 2.2: Função de Download Segura

In [3]:
import requests

def safe_download(url, timeout=10):
    """
    Baixa o conteúdo de uma URL de forma segura, com tratamento de erros HTTP.

    Args:
        url (str): A URL a ser baixada.
        timeout (int): O tempo limite em segundos para a requisição.

    Returns:
        requests.Response or None: O objeto Response em caso de sucesso, ou None em caso de falha.
    """
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()  # Levanta um HTTPError para respostas de erro (4xx ou 5xx)
        print(f"Download de {url} realizado com sucesso! Status: {response.status_code}")
        return response
    except requests.exceptions.HTTPError as http_err:
        print(f"Erro HTTP ao baixar {url}: {http_err}")
        print(f"Status Code: {response.status_code}, Razão: {response.reason}")
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Erro de conexão ao baixar {url}: {conn_err}")
    except requests.exceptions.Timeout as timeout_err:
        print(f"Tempo limite excedido ao baixar {url}: {timeout_err}")
    except requests.exceptions.RequestException as req_err:
        print(f"Um erro inesperado ocorreu ao baixar {url}: {req_err}")
    return None

# Demonstração da função com um URL que deve funcionar
print("\nTestando a função safe_download com um URL válido...")
valid_url = 'https://www.google.com'
response_valid = safe_download(valid_url)
if response_valid:
    print(f"Tamanho do conteúdo baixado: {len(response_valid.content)} bytes")

# Demonstração da função com um URL que deve falhar (ex: 404 Not Found)
print("\nTestando a função safe_download com um URL inválido (404 esperado)...")
invalid_url = 'https://www.google.com/nonexistent_page_12345'
response_invalid = safe_download(invalid_url)



Testando a função safe_download com um URL válido...
Download de https://www.google.com realizado com sucesso! Status: 200
Tamanho do conteúdo baixado: 84067 bytes

Testando a função safe_download com um URL inválido (404 esperado)...
Erro HTTP ao baixar https://www.google.com/nonexistent_page_12345: 404 Client Error: Not Found for url: https://www.google.com/nonexistent_page_12345
Status Code: 404, Razão: Not Found


### Exercício 2.3: Download e Salvamento de Imagem Aleatória

In [4]:
import requests

# URL para baixar uma imagem aleatória
image_url = 'https://picsum.photos/400/400'

# Nome do arquivo para salvar a imagem
file_name = 'random_image.jpg'

print(f"Baixando imagem de {image_url}...")

try:
    # Fazendo a requisição GET para a imagem com timeout
    response = requests.get(image_url, timeout=10)
    response.raise_for_status() # Lança exceções para códigos de status HTTP 4xx/5xx

    # Salvando o conteúdo bruto da resposta em um arquivo JPG
    with open(file_name, 'wb') as f:
        f.write(response.content)

    print(f"Imagem salva com sucesso em '{file_name}'")

except requests.exceptions.Timeout:
    print("A requisição da imagem excedeu o tempo limite (timeout).")
except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro ao baixar ou salvar a imagem: {e}")


Baixando imagem de https://picsum.photos/400/400...
Imagem salva com sucesso em 'random_image.jpg'


## Nível 3: Avançado (Webscraping e Ética)

### Exercício 3.1: Verificação do `robots.txt`

In [6]:
import requests

# URL base do site para verificar o robots.txt
site_url = 'https://books.toscrape.com/'
robots_url = site_url + 'robots.txt'

print(f"Verificando o arquivo robots.txt de: {site_url}")

try:
    response = requests.get(robots_url, timeout=10)
    response.raise_for_status() # Lança um erro para status codes ruins

    print("\nConteúdo do robots.txt:")
    print(response.text)

except requests.exceptions.Timeout:
    print(f"A requisição para {robots_url} excedeu o tempo limite (timeout).")
except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro ao acessar {robots_url}: {e}")


Verificando o arquivo robots.txt de: https://books.toscrape.com/
Ocorreu um erro ao acessar https://books.toscrape.com/robots.txt: 404 Client Error: Not Found for url: https://books.toscrape.com/robots.txt


### Exercício 3.2: Webscraping com BeautifulSoup (Extrair Título e Preço dos 5 primeiros livros)

In [7]:
import requests
from bs4 import BeautifulSoup

# URL do site para raspagem
scrape_url = 'https://books.toscrape.com/'

print(f"Iniciando webscraping em: {scrape_url}")

try:
    response = requests.get(scrape_url, timeout=10)
    response.raise_for_status() # Lança um erro para status codes ruins

    # Parseando o conteúdo HTML com BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    # Encontrando os livros (eles estão dentro de <article class='product_pod'>)
    books = soup.find_all('article', class_='product_pod')

    extracted_books = []
    for i, book in enumerate(books):
        if i >= 5: # Limitar aos 5 primeiros livros
            break

        # Extrair título
        title = book.h3.a['title']

        # Extrair preço
        # O preço está dentro de <p class='price_color'>
        price_str = book.find('p', class_='price_color').text.strip()
        # Remover o símbolo de moeda e converter para float
        price = float(price_str.replace('Â£', '')) # A página usa £, que pode ser renderizado como 'Â£'

        extracted_books.append({'title': title, 'price': price})

    print("\nDados dos 5 primeiros livros extraídos:")
    for book in extracted_books:
        print(f"Título: {book['title']}, Preço: £{book['price']:.2f}")

except requests.exceptions.Timeout:
    print(f"A requisição para {scrape_url} excedeu o tempo limite (timeout).")
except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro ao acessar {scrape_url}: {e}")
except Exception as e:
    print(f"Ocorreu um erro ao processar os dados: {e}")


Iniciando webscraping em: https://books.toscrape.com/

Dados dos 5 primeiros livros extraídos:
Título: A Light in the Attic, Preço: £51.77
Título: Tipping the Velvet, Preço: £53.74
Título: Soumission, Preço: £50.10
Título: Sharp Objects, Preço: £47.82
Título: Sapiens: A Brief History of Humankind, Preço: £54.23


### Exercício 3.3: Salvar dados extraídos dos livros em CSV com Pandas

In [8]:
import pandas as pd

# Supondo que `extracted_books` contém os dados do exercício anterior
# Se a célula anterior não foi executada ou a variável não existe,
# pode-se definir um exemplo para teste:
if 'extracted_books' not in locals():
    print("A variável 'extracted_books' não foi encontrada. Usando dados de exemplo.")
    extracted_books = [
        {'title': 'A Light in the Attic', 'price': 51.77},
        {'title': 'Tipping the Velvet', 'price': 53.74},
        {'title': 'Soumission', 'price': 50.10},
        {'title': 'Sharp Objects', 'price': 47.82},
        {'title': 'Sapiens: A Brief History of Humankind', 'price': 54.23}
    ]

# Criar DataFrame a partir dos dados extraídos
df_books = pd.DataFrame(extracted_books)

# Nome do arquivo CSV
csv_file_name = 'books_data.csv'

# Salvar DataFrame em CSV
df_books.to_csv(csv_file_name, index=False, encoding='utf-8')

print(f"\nDados dos livros salvos com sucesso em '{csv_file_name}'")
display(df_books.head())



Dados dos livros salvos com sucesso em 'books_data.csv'


,title,price
0,A Light in the Attic,51.77
1,Tipping the Velvet,53.74
2,Soumission,50.10
3,Sharp Objects,47.82
4,Sapiens: A Brief History of Humankind,54.23


### Exercício 3.4: Capturar Tabela da Wikipedia com `pandas.read_html()`

In [9]:
import pandas as pd
import requests
import io

# URL de uma página da Wikipedia com uma tabela
# Exemplo: Lista de países por população
wikipedia_url = 'https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o'

print(f"Tentando extrair tabelas de: {wikipedia_url}")

try:
    # Fazendo a requisição GET para a página da Wikipedia
    response = requests.get(wikipedia_url, timeout=15)
    response.raise_for_status() # Lança um erro para status codes ruins

    # Usar io.StringIO para que pandas.read_html possa ler o conteúdo como um arquivo
    html_content = io.StringIO(response.text)

    # Ler as tabelas HTML da página
    # read_html retorna uma lista de DataFrames, um para cada tabela encontrada
    tables = pd.read_html(html_content)

    print(f"Número de tabelas encontradas na página: {len(tables)}")

    if tables:
        # Exibir a primeira tabela encontrada (geralmente a mais relevante)
        df_wikipedia = tables[0]
        print("\nPrimeira tabela extraída:")
        display(df_wikipedia.head())
    else:
        print("Nenhuma tabela encontrada nesta página da Wikipedia.")

except requests.exceptions.Timeout:
    print(f"A requisição para {wikipedia_url} excedeu o tempo limite (timeout).")
except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro ao acessar {wikipedia_url}: {e}")
except Exception as e:
    print(f"Ocorreu um erro ao processar as tabelas HTML: {e}")


Tentando extrair tabelas de: https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o
Ocorreu um erro ao acessar https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o: 403 Client Error: Forbidden for url: https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o
